<a href="https://colab.research.google.com/github/alexzheng02/BST_strings/blob/master/Trends_in_Vehicle_Fuel_Efficiency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

How has the average fuel efficiency of new vehicles changed over time, and which vehicle classes have improved the most vs. least?

# Sustainability & Critical Infrastructure — Datathon 2026

---


### Problem Statement

Transportation is one of the leading sources of greenhouse gas emissions in the United States, with personal vehicles accounting for a significant share of national fuel consumption. As automakers respond to tightening federal regulations, rising fuel prices, and growing consumer demand for efficiency, the fuel economy of new vehicles has evolved considerably over recent decades. Understanding **how fuel efficiency has changed over time, and which segments of the vehicle market have made the greatest strides or fallen furthest behind**, is essential for evaluating the effectiveness of fuel economy standards, informing consumer policy, and identifying where the greatest opportunities for emissions reduction still remain.

**Research Question:** *How has the average fuel efficiency (MPG) of new vehicles changed over time, and which vehicle classes have improved the most vs. least?*

### Hypothesis

> **H₁:** Fuel efficiency (MPG) has improved **significantly** over time, but the rate of improvement varies across vehicle classes — with **passenger cars** showing the **greatest** gains and **trucks/SUVs** showing the **least**, due to sustained regulatory pressure on smaller vehicles and persistent consumer demand for larger ones.

###Methods
In this study, we are analyzing a dataset containing information about emission statistics, fuel types, MPG, and classes of vehicles. The data for these vehicles span over 35 years, from 1984 to 2019.

###Solutions   
By analyzing this dataset of vehicle information, we can:
*  **Identify long-term MPG** trends across the U.S. vehicle fleet from 1984 to 2019
*  Pinpoint which **vehicle classes** have improved the most and least
*  **Separate** conventional vehicles from alternative powertrains


###Benefits
The insights derived from this analysis provide value across multiple stakeholders:
*  **Consumers** gain a clearer picture of how fuel efficiency varies across vehicle types
*  **Policymakers** can use the findings to assess the historical impact of fuel economy regulations and design more targeted standards for vehicle classes that have shown the least improvement.
*  **Environmental advocates** can leverage the CO2 and MPG relationship findings to make a quantified case for stricter emissions standards
*  **Automakers** can benchmark their vehicle lineups against industry-wide trend

##Dataset Description
**Source:** [Vehicle Fuel Economy - Data.gov](https://catalog.data.gov/dataset/vehicle-fuel-economy
 )

 **Original provider:** Office of Energy Efficiency & Renewable Energy (EERE) via FuelEconomy.gov

The dataset contains vehicle fuel economy and emissions statistics, fuel/engine/drivetrain types, and other vehicle. Fuel economy data are the result of vehicle testing done at the EPA's National Vehicle and Fuel Emissions Laboratory in Ann Arbor, Michigan, and by vehicle manufacturers with EPA oversight. The dataset covers fuel economy datafiles from **1978 to 2019**.


#Data Cleaning & Preprocessing

###Importing Libraries

In [4]:
import matplotlib.pyplot as plt
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re


import geopandas as gpd
import plotly.express as px
from shapely.geometry import Point
import seaborn as sns
import missingno as msno

pd.set_option('display.max_columns', None)  # Display all columns

In [5]:
df = pd.read_csv('/vehicles.csv', low_memory=False)

# Select only the columns relevant to our issue
cols = {
    'year':             'Model Year',
    'make':             'Manufacturer',
    'model':            'Model',
    'VClass':           'Vehicle Class',
    'fuelType1':        'Fuel Type',
    'atvType':          'Alt. Powertrain',
    'comb08':           'Combined MPG',
    'city08':           'City MPG',
    'highway08':        'Highway MPG',
    'co2TailpipeGpm':   'CO2 Emissions (g/mile)',
}

df_clean = df[list(cols.keys())].rename(columns=cols)

pd.set_option('display.max_columns', None)
display(df_clean.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '/vehicles.csv'

###Missing Values

Before analyzing the data, we need to check for any missing values or unexpected inputs in the columns we're focusing on.

In [ ]:
# Missing values
missing = df.isnull().sum().rename('Missing').to_frame()
missing['% Missing'] = (missing['Missing'] / len(df) * 100).round(2)
missing['Dtype'] = df.dtypes.values

# Filter for relevant columns and then rename the index
relevant_missing = missing.loc[list(cols.keys())]
relevant_missing = relevant_missing.rename(index=cols)

print('── Missing Value Summary ──────────────────────')
display(relevant_missing)

# Basic stats on MPG columns
print('\n── MPG Descriptive Statistics ─────────────────')
display(df_clean[['Combined MPG','City MPG','Highway MPG','CO2 Emissions (g/mile)']].describe().round(1))

## Investigating 'Alt. Powertrain' Missing Values

Let's confirm if the missing values in 'Alt. Powertrain' correspond to conventional gasoline vehicles. We will first look at the unique values present in the 'Alt. Powertrain' column to understand its categories, and then filter the DataFrame to inspect the 'Fuel Type' of vehicles where 'Alt. Powertrain' is missing.

In [ ]:
# Check unique values in 'Alt. Powertrain'
print("Unique values in 'Alt. Powertrain':")
display(df_clean['Alt. Powertrain'].value_counts(dropna=False))

# Filter for rows where 'Alt. Powertrain' is missing and check 'Fuel Type'
missing_alt_powertrain = df_clean[df_clean['Alt. Powertrain'].isna()]

print("\nFuel Types for vehicles with missing 'Alt. Powertrain':")
display(missing_alt_powertrain['Fuel Type'].value_counts())

### Findings
The results confirm that the overwhelming majority of vehicles with a missing Alt. Powertrain value run on Regular, Premium, or Diesel gasoline. These are all conventional fuel types. This means the missing values are not random, but are structurally missing because these vehicles do not have an alternative powertrain. It is therefore safe to fill these nulls with the label 'Conventional' rather than dropping them, preserving all records in the dataset.

In [ ]:
# ── Fill Alt. Powertrain nulls ────────────────────────────────────
# Confirmed: missing = conventional gasoline vehicles
df_clean['Alt. Powertrain'] = df_clean['Alt. Powertrain'].fillna('Conventional')

print("Updated value counts:")
display(df_clean['Alt. Powertrain'].value_counts())

### Filtering

We want to filter out gas and diesel for the analysis. EV's and Hybrid vehicles will be analyzed seperately, as their MPGe would affect the trend of the MPGs.

In [ ]:
GAS_FUELS = ['Regular Gasoline', 'Premium Gasoline', 'Diesel', 'Midgrade Gasoline']

df_gas  = df_clean[df_clean['Fuel Type'].isin(GAS_FUELS) & (df_clean['Combined MPG'] > 0) & (df_clean['Alt. Powertrain'] == 'Conventional')].copy()
df_alt  = df_clean[df_clean['Alt. Powertrain'].isin(['Hybrid','EV','Plug-in Hybrid'])].copy()

print(f'Gas/Diesel records : {len(df_gas):,}')
print(f'Hybrid/EV records  : {len(df_alt):,}')
print(f'Dropped (other)   : {len(df_clean) - len(df_gas) - len(df_alt):,}')

## Efficiency Trend Across All Vehicles
Shows the combined MPG of gas and diesel vehicles over time.
This section analyzes the overall fuel efficiency trend of gas and diesel vehicles from 1984 to 2019. By plotting both the mean and median Combined MPG per year, we can identify long-term improvement patterns while accounting for outliers that may skew the average.

In [ ]:
yearly = df_gas.groupby('Model Year').agg(
    mean_mpg  = ('Combined MPG', 'mean'),
    median_mpg= ('Combined MPG', 'median'),
    count     = ('Combined MPG', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(yearly['Model Year'], yearly['mean_mpg'],
        color=PALETTE[0], linewidth=2.5, label='Mean MPG', zorder=3)
ax.plot(yearly['Model Year'], yearly['median_mpg'],
        color=PALETTE[3], linewidth=1.8, linestyle='--', label='Median MPG', zorder=3)
ax.fill_between(yearly['Model Year'], yearly['mean_mpg'],
                alpha=0.12, color=PALETTE[0])

# Annotate key years
for yr, label, dy in [(1984,'CAFE tightened\n(1984)',1.5),(2008,'Financial\ncrisis',1.5),(2016,'Peak efficiency',1.5)]:
    row = yearly[yearly['Model Year']==yr]
    if not row.empty:
        ax.annotate(label,
                    xy=(yr, row['mean_mpg'].values[0]),
                    xytext=(yr, row['mean_mpg'].values[0]+dy),
                    fontsize=7.5, color='#555',
                    arrowprops=dict(arrowstyle='->', color='#aaa', lw=0.8),
                    ha='center')

ax.set_title('Average Combined MPG of New Gas/Diesel Models — 1984 to 2019',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Model Year', fontsize=10)
ax.set_ylabel('Average Combined MPG', fontsize=10)
ax.legend(fontsize=9)
# Check if yearly DataFrame is empty before setting x-axis major locator
if not yearly.empty:
    import matplotlib.ticker as mticker
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

# Check if yearly DataFrame is empty before calculating gain
if not yearly.empty:
    gain = yearly['mean_mpg'].iloc[-1] - yearly['mean_mpg'].iloc[0]
    print(f'Fleet average improved by {gain:.1f} MPG from {yearly["Model Year"].iloc[0]} to {yearly["Model Year"].iloc[-1]}')
else:
    print("Cannot calculate gain: yearly DataFrame is empty.")

###Key Findings
The trend line reveals three distinct phases in fuel efficiency:

*  1984–1990: An initial rise in average MPG driven by the tightening of federal CAFE (Corporate Average Fuel Economy) standards, which forced automakers to produce more efficient vehicles following the 1970s energy crisis.
*  1990–2004: A period of relative stagnation where MPG plateaued, coinciding with the explosive growth in consumer demand for SUVs and light trucks — vehicle types that were held to lower fuel economy standards.
*  2005–2016: A strong rebound in efficiency driven by updated CAFE standards, rising fuel prices, and advances in engine technology such as turbocharging, direct injection, and dual-clutch transmissions. The fleet average peaked around 2016 before slightly declining as trucks and SUVs continued to dominate sales.

## Average MPG by Class
Shows change in MPG by individual vehicle class for the most common vehicles. While the overall trend provides a fleet-wide view, it masks significant variation across vehicle classes. This chart breaks down average Combined MPG by the five most common vehicle classes over time, allowing us to identify which segments drove efficiency gains and which lagged behind.

In [ ]:
# Group by Model Year and Vehicle Class to see trends for each class
yearly_class_mpg = df_gas.groupby(['Model Year', 'Vehicle Class'])['Combined MPG'].mean().reset_index()

# Identify top vehicle classes to plot for clarity (e.g., top 5 by count or most common ones)
top_classes = df_gas['Vehicle Class'].value_counts().nlargest(5).index.tolist()

fig, ax = plt.subplots(figsize=(15, 7))

for i, v_class in enumerate(top_classes):
    class_data = yearly_class_mpg[yearly_class_mpg['Vehicle Class'] == v_class]
    ax.plot(class_data['Model Year'], class_data['Combined MPG'], label=v_class, color=PALETTE[i], linewidth=2)

ax.set_title('Average Combined MPG by Vehicle Class — 1984 to 2019',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Model Year', fontsize=12)
ax.set_ylabel('Average Combined MPG', fontsize=12)
ax.legend(title='Vehicle Class', fontsize=10, title_fontsize=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
ax.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

###Key Findings:

*  **Compact and midsize cars** consistently maintained the highest MPG ratings throughout the study period, benefiting most from regulatory pressure and lightweight design improvements.
*  **SUVs and pickup trucks** started at the lowest MPG levels and showed the slowest rate of improvement, reflecting both their heavier weight and the historically looser CAFE standards applied to light trucks.
*  **The gap between car classes and truck/SUV classes** narrowed slightly after 2011, when the Obama administration extended CAFE standards to include light trucks more aggressively — but a meaningful disparity remained through 2019.
*  **All classes** show a noticeable dip around **2008–2009**, which aligns with the financial crisis reducing new model investment, followed by a sharp recovery as automakers modernized their lineups.

##MPG Improvement by Vehicle Class
Shows the improvement from highest to lowest by type of vehicle. To directly answer our research hypothesis, we calculate the net MPG improvement for each vehicle class by comparing its earliest recorded average MPG to its most recent. This ranking reveals which classes made the greatest efficiency gains over the study period and which improved the least.


In [ ]:
class_improvements = yearly_class_mpg.groupby('Vehicle Class').agg(
    first_mpg=('Combined MPG', lambda x: x.iloc[0]),
    last_mpg=('Combined MPG', lambda x: x.iloc[-1])
).reset_index()

class_improvements['MPG_Improvement'] = class_improvements['last_mpg'] - class_improvements['first_mpg']

# Sort by improvement for better visualization
class_improvements_sorted = class_improvements.sort_values(by='MPG_Improvement', ascending=False)

fig, ax = plt.subplots(figsize=(15, 8))
sns.barplot(x='MPG_Improvement', y='Vehicle Class', data=class_improvements_sorted, palette='viridis', ax=ax)

ax.set_title('MPG Improvement by Vehicle Class (1984-2019)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Combined MPG Improvement', fontsize=12)
ax.set_ylabel('Vehicle Class', fontsize=12)
ax.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

###Key Findings:
*  The **bar chart** confirms our hypothesis partially — improvement is uneven across vehicle classes, but the results are more nuanced than expected. Vehicle classes with the **greatest improvement** tend to be those that started with relatively low baseline MPG, giving them more room to grow under regulatory pressure.
* **Pickup trucks and large SUVs** appear toward the **bottom** of the ranking, consistent with our hypothesis that consumer demand for larger vehicles has offset potential efficiency gains.
* **Negative or near-zero improvement** in some classes may reflect a composition effect — as those classes grew to include larger, heavier models over time, the average MPG was pulled down despite engine-level improvements.
* Overall, this visualization is the most direct evidence for our alternative hypothesis (**H₁**) — that MPG improvement rates vary significantly by vehicle class.

## Efficiency and CO2 Emissions Correlation
This section examines the relationship between fuel efficiency (Combined MPG) and tailpipe CO2 emissions (g/mile) from two angles — a scatter plot showing the direct correlation between the two variables, and a time series showing how average CO2 emissions have changed across new vehicle models from 1984 to 2019.


In [ ]:
  co2_df = df_gas[df_gas['CO2 Emissions (g/mile)'] > 0].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter MPG vs CO2
ax = axes[0]
sample = co2_df.sample(min(3000, len(co2_df)), random_state=42)
ax.scatter(sample['Combined MPG'], sample['CO2 Emissions (g/mile)'],
           alpha=0.25, s=12, color=PALETTE[0], rasterized=True)
# Fit line
m, b = np.polyfit(co2_df['Combined MPG'], co2_df['CO2 Emissions (g/mile)'], 1)
x_line = np.linspace(co2_df['Combined MPG'].min(), co2_df['Combined MPG'].max(), 100)
ax.plot(x_line, m*x_line+b, color=PALETTE[3], linewidth=2, label=f'y = {m:.1f}x + {b:.0f}')
ax.set_title('Combined MPG vs. CO₂ Emissions', fontsize=11, fontweight='bold')
ax.set_xlabel('Combined MPG')
ax.set_ylabel('CO₂ Tailpipe (g/mile)')
ax.legend(fontsize=9)

# Right: avg CO2 by year
ax2 = axes[1]
co2_yearly = co2_df.groupby('Model Year')['CO2 Emissions (g/mile)'].mean()
ax2.plot(co2_yearly.index, co2_yearly.values, color=PALETTE[3], linewidth=2.5)
ax2.fill_between(co2_yearly.index, co2_yearly.values, alpha=0.12, color=PALETTE[3])
ax2.set_title('Average CO₂ Emissions of New Models Over Time', fontsize=11, fontweight='bold')
ax2.set_xlabel('Model Year')
ax2.set_ylabel('Avg CO₂ (g/mile)')
ax2.xaxis.set_major_locator(mticker.MultipleLocator(5))

plt.tight_layout()
plt.show()

## Key Findings
**Left Plot — MPG vs CO2 Scatter:**
*  **Strong negative linear relationship** — the fitted regression line confirms that as Combined MPG increases, CO2 tailpipe emissions decrease in a consistent and predictable pattern. The equation y = mx + b printed on the chart quantifies exactly how many grams of CO2 are reduced per additional MPG gained.
*  **The relationship is tight but not perfectly linear** — the scatter of points around the trend line widens at lower MPG values, indicating that low-efficiency vehicles vary more in their emissions output, likely due to differences in engine size, fuel type, and vehicle weight.
*  **Most vehicles cluster between 15–35 MPG** — reflecting the dominance of conventional gas-powered cars, trucks, and SUVs in the dataset, with a smaller tail of higher-efficiency models extending toward 50+ MPG.

**Right Plot — CO2 Over Time:**
*  Mirroring the MPG improvement trend identified earlier, average tailpipe CO2 emissions **fell sharply** from the mid-2000s onward
*  **Emissions plateaued between 1990–2004** — consistent with the MPG stagnation period driven by the SUV boom, where efficiency gains in passenger cars were offset by the growing dominance of less efficient light trucks.
*  The **downward trend after 2005** represents meaningful environmental progress — the reduction in average CO2 per mile across new models translates directly to **lower greenhouse gas output** at the fleet level, validating fuel economy standards as an effective emissions reduction tool.

## Alternate Powertrains Usage
While our primary analysis focuses on conventional gas and diesel vehicles, it is important to contextualize the rise of alternative powertrain technologies over the same period. This chart tracks the number of hybrid, plug-in hybrid, and electric vehicle models introduced per year, illustrating how the alternative vehicle market has grown alongside — and in response to — the same regulatory and economic pressures driving conventional MPG improvements.

In [ ]:
atv_yearly = (
    df_alt
    .groupby(['Model Year','Alt. Powertrain'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(13, 5))
atv_yearly.plot.area(ax=ax, alpha=0.75,
                     color=PALETTE[:len(atv_yearly.columns)],
                     linewidth=0)
ax.set_title('Alternative Powertrain Models Introduced per Year',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Model Year', fontsize=10)
ax.set_ylabel('Number of Models', fontsize=10)
ax.legend(title='Powertrain', fontsize=9, title_fontsize=9)
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

###Key Findings:

*  **Hybrids dominated early adoption** — hybrid models were the first alternative powertrain to appear in meaningful numbers, beginning their rise in the early 2000s following the introduction of pioneering models. Their growth reflects early consumer and manufacturer interest in fuel savings without the range limitations of fully electric vehicles.
*  **Electric and Plug-in Hybrid vehicles emerged later but grew rapidly** — EVs and PHEVs remained nearly absent until around 2010–2011, after which their numbers increased sharply. This aligns with the federal tax incentives introduced under the Energy Improvement and Extension Act and growing investment in battery technology.
*  **The overall scale remains small relative to conventional vehicles** — with 874 alternative powertrain records compared to 37,217 gas/diesel records, alternative vehicles represent a small but rapidly growing share of new models introduced. This justifies our decision to analyze them separately, as including their MPGe values would significantly distort conventional MPG trend comparisons.

###Conclusion and Future Work

The dataset only tracks models offered, not models sold. During the same period that fuel economy improved on paper, consumers shifted heavily toward trucks and SUVs. So while automakers were offering more efficient vehicles, the average car actually on the road may not have improved as much as the dataset implies. The efficiency gains were real, but they were partially cancelled out by changing consumer preferences.

In future work, the dataset should be extended to the present, such that current rapid advancements in technology and spread of EV usage can be accounted for. It is important to acknowledge that because the dataset ends in 2019, trends may not be completely accurate in current-day data. Future work should also track sales-weighted data and divide data by region to get a more accurate depiction of changes in efficiency.

###Dataset Citations (MLA 8)

California Air Resources Board. Vehicle Fuel Economy. State of California, 7 Dec. 2018, catalog.data.gov/dataset/vehicle-fuel-economy. Accessed 29 Mar. 2026.